# AutoEDA de precios de maíz blanco — SNIIM

## 1 | Objetivo y alcance

Obtener un panorama inicial de la estructura, los datos faltantes y las
distribuciones de los precios del maíz blanco registrados por el SNIIM
durante 2013–2025.

Cada fila representa una combinación de mes, mercado y origen.
El precio se expresa en pesos mexicanos por kilogramo.

Conservamos los precios faltantes y mantenemos separados los distintos
orígenes. Un precio faltante para un origen no implica necesariamente
que el mercado completo carezca de precios en ese mes.

El reporte automático complementa la exploración manual de la libreta
`01-eda-sniim-cobertura-faltantes.ipynb`. Sus alertas requieren revisión
y no constituyen evidencia de errores ni de relaciones causales.

In [1]:
# Verificamos el entorno del autoEDA.
import sys
import pandas as pd
from data_profiling import ProfileReport

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

C:\Users\omicr\MCD\CURSOS\ingcar20262\Proyecto1\.venv-autoeda-sniim\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.14
Pandas: 2.3.3


## 2 | Carga y revisión de los datos

¿Qué contiene el archivo procesado que utilizaremos para el autoEDA?

Cargamos la tabla por mes, mercado y origen, sin sustituir los precios
faltantes ni promediar los distintos orígenes.

In [2]:
from pathlib import Path

# Ubicamos la raíz desde el repositorio o la carpeta notebooks.
raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

# Cargamos la tabla original y reconocemos la fecha.
ruta = raiz / "data" / "processed" / "sniim_maiz_blanco_mensual.csv"
datos = pd.read_csv(ruta, parse_dates=["fecha"])

# Revisamos dimensiones, columnas y primeras observaciones.
print("Filas y columnas:", datos.shape)
display(datos.head())
datos.info()

Filas y columnas: (960, 6)


,fecha,estado_mercado,mercado,origen,producto,precio_mxn_kg
0,2013-01-01,Sonora,Hermosillo,Sinaloa,Maíz blanco,6.36
1,2013-02-01,Sonora,Hermosillo,Sinaloa,Maíz blanco,6.00
2,2013-03-01,Sonora,Hermosillo,Sinaloa,Maíz blanco,6.50
3,2013-04-01,Sonora,Hermosillo,Sinaloa,Maíz blanco,6.24
4,2013-05-01,Sonora,Hermosillo,Sinaloa,Maíz blanco,5.62


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   fecha           960 non-null    datetime64[ns]
 1   estado_mercado  960 non-null    object        
 2   mercado         960 non-null    object        
 3   origen          960 non-null    object        
 4   producto        960 non-null    object        
 5   precio_mxn_kg   791 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 45.1+ KB


## 3 | Generación del reporte automático

¿Qué patrones y alertas identifica la herramienta en la tabla original?

El reporte resume los tipos de variables, sus distribuciones, los valores
faltantes y las asociaciones que puede calcular.

Las alertas son puntos de partida para revisar, no instrucciones para
eliminar o transformar datos. Sus resultados describen filas por mes,
mercado y origen, no una tabla con un único precio por mercado y mes.

In [3]:
# Configuramos el reporte sobre los datos sin imputación ni agregación.
reporte = ProfileReport(
    datos,
    title="AutoEDA SNIIM | Precios de maíz blanco, 2013–2025",
    explorative=True,
    pool_size=1,
)

# Guardamos el HTML junto a los reportes automáticos del equipo.
carpeta_reportes = raiz / "reports" / "auto_eda"
carpeta_reportes.mkdir(parents=True, exist_ok=True)

ruta_reporte = carpeta_reportes / "sniim_maiz_blanco.html"
reporte.to_file(ruta_reporte)

print("Reporte guardado en:", ruta_reporte)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 83.42it/s]

Reporte guardado en: C:\Users\omicr\MCD\CURSOS\ingcar20262\Proyecto1\Rendimiento_Bursatil_Sector_Agropecuario\reports\auto_eda\sniim_maiz_blanco.html


## 4 | Interpretación de los resultados automáticos

El reporte identifica 169 precios faltantes, equivalentes al 17.6 %
de los registros. El 2.9 % mostrado en el resumen general utiliza
como denominador todas las celdas del conjunto.

La variable `producto` es constante porque el análisis se limita al
maíz blanco. La asociación entre `estado_mercado` y `mercado`
corresponde a la ubicación de los mercados seleccionados.
Estas alertas no justifican eliminar columnas automáticamente.

Las frecuencias de mercados y orígenes representan filas registradas,
no cantidades comercializadas ni participación en la producción.

Los precios observados van de 3.50 a 16.25 MXN/kg, sin valores cero,
negativos o infinitos. Esto no garantiza que todos sean correctos:
los cambios extremos requieren revisión contextual.

El reporte no distingue directamente entre un origen sin precio y
un mercado sin ningún precio durante un mes. Esa distinción se
desarrolla en la libreta 01, donde se identificaron 12 combinaciones
mercado–mes sin precio disponible.

Las asociaciones automáticas no sustituyen el análisis temporal
ni permiten establecer relaciones causales.

## 5 | Verificación de la unidad de observación

¿Existe más de una fila para la misma combinación de mes, mercado y origen?

Revisamos la clave que identifica cada observación. Tener varios
orígenes en un mismo mercado y mes es válido; repetir la misma
combinación requiere inspección.

In [4]:
# Identificamos todas las filas con una clave repetida.
clave = ["fecha", "mercado", "origen"]
claves_repetidas = datos.duplicated(subset=clave, keep=False)

# Contamos las filas involucradas y mostramos los casos.
print("Filas con clave repetida:", claves_repetidas.sum())

if claves_repetidas.any():
    display(
        datos.loc[claves_repetidas]
        .sort_values(clave)
    )
else:
    print("Cada combinación de mes, mercado y origen es única.")

Filas con clave repetida: 0
Cada combinación de mes, mercado y origen es única.


### 5.1 | Resultado de la verificación

No se encontraron combinaciones repetidas de fecha, mercado y origen.
Esto respalda la unidad de observación definida para la tabla.

La presencia de varios orígenes en un mismo mercado y mes no constituye
una duplicación. Esta comprobación no demuestra que la cobertura esté
completa ni que todos los precios sean correctos.

## 6 | Síntesis y vínculo con la exploración manual

El reporte automático confirma que el conjunto contiene 960 registros,
seis columnas y 169 precios faltantes. No hay filas completamente
duplicadas ni claves mes–mercado–origen repetidas.

Las alertas sobre el producto constante y la asociación entre estado
y mercado se explican por la selección y estructura del conjunto.
No requieren modificar los datos.

Conservamos los precios faltantes y los registros por origen.
La disponibilidad mensual, las variaciones, las anomalías y las
relaciones entre mercados se estudian en la libreta
`01-eda-sniim-cobertura-faltantes.ipynb`.

Como seguimiento de la consulta con el experto, queda pendiente
explorar patrones por mes del año y revisar cambios en los orígenes,
sin atribuirles por sí solos causas económicas.